# Σημειώσεις Pandas: Οριζόντια Σύνδεση Πινάκων (`Inner Merge`)

Η συνάρτηση `pd.merge()` συνδέει δύο DataFrames **οριζόντια** (προσθέτει στήλες) με βάση μια κοινή στήλη-κλειδί (Primary Key & Foreign Key).

## 💡 Η Λογική του Inner Merge («Και οι δύο πλευρές πρέπει να ταιριάζουν»)
* Είναι η **προεπιλεγμένη (default)** συμπεριφορά του `pd.merge()` (`how="inner"`).
* **Κρατάει ΜΟΝΟ** τις γραμμές των οποίων το κλειδί υπάρχει **και στους δύο πίνακες**.
* Αν μια εγγραφή του αριστερού πίνακα δεν βρει ταίρι στον δεξιό (ή το αντίστροφο), **διαγράφεται** από το τελικό αποτέλεσμα.

---

## 🔍 Η Βασική Σύνταξη (2 Περιπτώσεις)

### 1. Όταν οι στήλες-κλειδιά έχουν ΔΙΑΦΟΡΕΤΙΚΟ όνομα (`left_on` & `right_on`)
```python
import pandas as pd

result = pd.merge(
    employees,          # Αριστερός πίνακας (left_df)
    departments,        # Δεξιός πίνακας (right_df)
    left_on="dept_id",  # Το κλειδί στον αριστερό πίνακα (Foreign Key)
    right_on="id"       # Το κλειδί στον δεξιό πίνακα (Primary Key)
)

# Επιλογή μόνο των στηλών που μας ενδιαφέρουν μετά την ένωση
result[["first_name", "dept_name", "salary"]]
```

### 2. Όταν οι στήλες-κλειδιά έχουν το ΙΔΙΟ όνομα (`on`)
```python
result = pd.merge(orders, customers, on="customer_id")
```

---

## 📌 Γρήγορος Κανόνας
* **`pd.concat()`** ➡️ Προσθέτει **γραμμές** (κάθετα).
* **`pd.merge()`** ➡️ Προσθέτει **στήλες** (οριζόντια).


In [21]:
import pandas as pd
import numpy as np

# 1. Πίνακας Πελατών (Primary Key: 'id')
# Σημείωση: Ο πελάτης 104 (Δημήτρης) δεν έχει κάνει καμία παραγγελία!
customers = pd.DataFrame({
    'id':[101, 102, 103, 104],
    'name': ['Maria', 'Nikos', 'Eleni', 'Dimitris'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze']
})

# 2. Πίνακας Παραγγελιών (Foreign Key: 'cust_id')
# Σημείωση: Η παραγγελία ORD-5 ανήκει στον πελάτη 999 που δεν υπάρχει στον πίνακα customers!
orders = pd.DataFrame({
    'order_id': ['ORD-1', 'ORD-2', 'ORD-3', 'ORD-4', 'ORD-5'],
    'cust_id':[101, 101, 103, 102, 999],
    'amount': [120.50, 45.00, 89.90, 210.00, 30.00]
})


In [22]:
customers

,id,name,tier
0,101,Maria,Gold
1,102,Nikos,Silver
2,103,Eleni,Gold
3,104,Dimitris,Bronze


In [23]:
orders

,order_id,cust_id,amount
0,ORD-1,101,120.5
1,ORD-2,101,45.0
2,ORD-3,103,89.9
3,ORD-4,102,210.0
4,ORD-5,999,30.0


In [24]:
# το how='inner' είναι προαιρετικό, από default είναι inner
combined = pd.merge(customers, orders, how='inner', left_on='id', right_on='cust_id')
combined = combined.drop(columns='cust_id') # δεν χρειάζεται να υπάρχει 2 φορές 
combined

,id,name,tier,order_id,amount
0,101,Maria,Gold,ORD-1,120.5
1,103,Eleni,Gold,ORD-3,89.9
2,101,Maria,Gold,ORD-2,45.0
3,102,Nikos,Silver,ORD-4,210.0


In [32]:
combined['amount'].mean()

np.float64(116.35)

In [ ]:
(
    combined.groupby(['id','name'])['amount']   #never group only by name, always by id
    .agg(['count','sum'])
    .sort_values('sum', ascending=False)
)

,,count,sum
id,name,,
102,Nikos,1,210.0
101,Maria,2,165.5
103,Eleni,1,89.9
